In [0]:
#Load Data
from pyspark.sql.functions import *
from pyspark.sql.window import Window

telemetry = spark.table(
    "workspace.predictive_maintenance.silver_telemetry"
)

In [0]:
#Select One Machine
machine1 = telemetry.filter(
    col("machineID") == 1
).orderBy("datetime")

display(machine1.limit(10))

datetime,machineID,volt,rotate,pressure,vibration
2015-01-01T06:00:00.000Z,1,176.217853015625,418.504078221616,113.077935462083,45.0876857639276
2015-01-01T07:00:00.000Z,1,162.87922289706,402.747489565395,95.4605253823187,43.4139726834815
2015-01-01T08:00:00.000Z,1,170.989902405567,527.349825452291,75.2379048586662,34.1788471214451
2015-01-01T09:00:00.000Z,1,162.462833264092,346.149335043074,109.248561276504,41.1221440884256
2015-01-01T10:00:00.000Z,1,157.61002119306,435.376873016938,111.886648210168,25.9905109982024
2015-01-01T11:00:00.000Z,1,172.504839196295,430.323362106675,95.9270416939636,35.6550173268837
2015-01-01T12:00:00.000Z,1,156.556030606329,499.071623068962,111.755684290096,42.7539196974773
2015-01-01T13:00:00.000Z,1,172.522780814836,409.624717000438,101.00108276407,35.4820086610704
2015-01-01T14:00:00.000Z,1,175.324523915223,398.648780707752,110.624360548654,45.4822868466294
2015-01-01T15:00:00.000Z,1,169.218423246933,460.850669930244,104.848229967003,39.9017354356787


In [0]:
#Create Moving Average Forecast
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Create window specification
window_spec = (
    Window
    .partitionBy("machineID")
    .orderBy("datetime")
    .rowsBetween(-5, -1)
)

# Calculate Moving Average Forecast
forecast_df = machine1.withColumn(
    "forecast_volt",
    avg("volt").over(window_spec)
)

# Display Results
display(
    forecast_df.select(
        "datetime",
        "machineID",
        "volt",
        "forecast_volt"
    )
)

datetime,machineID,volt,forecast_volt
2015-01-01T06:00:00.000Z,1,176.217853015625,null
2015-01-01T07:00:00.000Z,1,162.87922289706,176.217853015625
2015-01-01T08:00:00.000Z,1,170.989902405567,169.5485379563425
2015-01-01T09:00:00.000Z,1,162.462833264092,170.02899277275068
2015-01-01T10:00:00.000Z,1,157.61002119306,168.13745289558602
2015-01-01T11:00:00.000Z,1,172.504839196295,166.03196655508083
2015-01-01T12:00:00.000Z,1,156.556030606329,165.28936379121484
2015-01-01T13:00:00.000Z,1,172.522780814836,164.0247253330686
2015-01-01T14:00:00.000Z,1,175.324523915223,164.33130101492242
2015-01-01T15:00:00.000Z,1,169.218423246933,166.9036391451486


In [0]:
#Calculate Forecast Error
from pyspark.sql.functions import abs, col

forecast_accuracy = forecast_df.withColumn(
    "error",
    abs(col("volt") - col("forecast_volt"))
)

display(
    forecast_accuracy.select(
        "datetime",
        "volt",
        "forecast_volt",
        "error"
    )
)

datetime,volt,forecast_volt,error
2015-01-01T06:00:00.000Z,176.217853015625,null,null
2015-01-01T07:00:00.000Z,162.87922289706,176.217853015625,13.338630118564993
2015-01-01T08:00:00.000Z,170.989902405567,169.5485379563425,1.4413644492244941
2015-01-01T09:00:00.000Z,162.462833264092,170.02899277275068,7.5661595086586715
2015-01-01T10:00:00.000Z,157.61002119306,168.13745289558602,10.52743170252603
2015-01-01T11:00:00.000Z,172.504839196295,166.03196655508083,6.472872641214167
2015-01-01T12:00:00.000Z,156.556030606329,165.28936379121484,8.73333318488585
2015-01-01T13:00:00.000Z,172.522780814836,164.0247253330686,8.498055481767409
2015-01-01T14:00:00.000Z,175.324523915223,164.33130101492242,10.993222900300594
2015-01-01T15:00:00.000Z,169.218423246933,166.9036391451486,2.3147841017844257


In [0]:
#Average Forecast Error
from pyspark.sql.functions import avg

forecast_accuracy.select(
    avg("error").alias("mean_absolute_error")
).show()

+-------------------+
|mean_absolute_error|
+-------------------+
| 13.091359859657391|
+-------------------+

